# UCSD Dataset Setup

## Environment Setup
1. Mounts Google Drive for persistent file management for the dataset zip
2. Makes Git repository remain available across Colab runtime sessions.

In [54]:
from google.colab import drive
drive.mount('/content/drive')

import torch

if torch.cuda.is_available():
  props = torch.cuda.get_device_properties(0)
  print(f"GPU: {props.name} | VRAM: {props.total_memory / 1024**3} GB")
else:
  print("No GPU assigned")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
No GPU assigned


## GitHub Repository Setup

In [55]:
%cd "{REPO_DIR}"

!git config user.name "Rishabh G. Shetye"
!git config user.email "142067418+Rishabh-G-Shetye@users.noreply.github.com"

!git status

/content/drive/MyDrive/SurveillanceAnomalyDetection/repo
On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore
	.gitignore.gdoc
	README.md
	README.md.gdoc
	notebooks/

nothing added to commit but untracked files present (use "git add" to track)


## Dataset Configuration
1. The original UCSD dataset ZIP is stored permanently on the Google Drive
2. Extracted dataset is stored in /content because Colab's local storage is faster.
3. However /content is temporary and is cleaned when runtime expires or resets.

In [56]:
from google.colab import userdata
from pathlib import Path

GH_TOKEN = userdata.get("GH_PAT")

REPO_DIR = Path(
    "/content/drive/MyDrive/SurveillanceAnomalyDetection/repo"
)

REPO_URL = (
    f"https://{GH_TOKEN}@github.com/"
    "Rishabh-G-Shetye/SurveillanceAnomalyDetection.git"
)

if not REPO_DIR.exists():
    !git clone {REPO_URL} "{REPO_DIR}"
    print("Repository cloned.")
else:
    print("Repository already exists — skipping clone.")

%cd "{REPO_DIR}"

Repository already exists — skipping clone.
/content/drive/MyDrive/SurveillanceAnomalyDetection/repo


## Dataset Extraction

In [57]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

DATA_ROOT = Path("/content/UCSD_Anomaly_Dataset/UCSD_Anomaly_Dataset")

IGNORE_NAMES = {".DS_Store", "._.DS_Store"}
IGNORE_SUFFIXES = {".m", ".m~", ".txt", ".txt~"}

In [58]:
from zipfile import ZipFile

ZIP_PATH = Path("/content/drive/MyDrive/SurveillanceAnomalyDetection/UCSD_Anomaly_Dataset.zip")
EXTRACT_PATH = Path("/content/UCSD_Anomaly_Dataset")

PED1_PATH = DATA_ROOT / "UCSDped1"
PED2_PATH = DATA_ROOT / "UCSDped2"

if not (PED1_PATH.exists() and PED2_PATH.exists()):
  with ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_PATH)

print("Dataset at:", DATA_ROOT)
print("Ped1: ", PED1_PATH.exists())
print("Ped2: ", PED2_PATH.exists())

Dataset at: /content/UCSD_Anomaly_Dataset/UCSD_Anomaly_Dataset
Ped1:  True
Ped2:  True


## Dataset Structure Verification

In [59]:
def print_tree(root: Path, max_depth: int = 3, _depth: int = 0) -> None:
    """
    Print the directory structure under root upto max_depth
    This is used to verify the extracted UCSD dataset structure
    """
    if not root.exists():
        print(f"{root} does not exist -- check DATA_ROOT / that Drive is mounted")
        return
    for p in sorted(root.iterdir()):
        print("  " * _depth + p.name + ("/" if p.is_dir() else ""))
        if p.is_dir() and _depth < max_depth:
            print_tree(p, max_depth, _depth + 1)

print_tree(DATA_ROOT, max_depth=2)

.DS_Store
._README.txt
README.txt
README.txt~
UCSDped1/
  .DS_Store
  Test/
    .DS_Store
    ._UCSDped1.m
    Test001/
    Test002/
    Test003/
    Test003_gt/
    Test004/
    Test004_gt/
    Test005/
    Test006/
    Test007/
    Test008/
    Test009/
    Test010/
    Test011/
    Test012/
    Test013/
    Test014/
    Test014_gt/
    Test015/
    Test016/
    Test017/
    Test018/
    Test018_gt/
    Test019/
    Test019_gt/
    Test020/
    Test021/
    Test021_gt/
    Test022/
    Test022_gt/
    Test023/
    Test023_gt/
    Test024/
    Test024_gt/
    Test025/
    Test026/
    Test027/
    Test028/
    Test029/
    Test030/
    Test031/
    Test032/
    Test032_gt/
    Test033/
    Test034/
    Test035/
    Test036/
    UCSDped1.m
    UCSDped1.m~
  Train/
    .DS_Store
    ._.DS_Store
    Train001/
    Train002/
    Train003/
    Train004/
    Train005/
    Train006/
    Train007/
    Train008/
    Train009/
    Train010/
    Train011/
    Train012/
    Train013/
    Train014/

## Image check

In [60]:
train_path = PED1_PATH / "Train"

for path in train_path.rglob("*"):
    if (
        path.is_file()
        and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
    ):
        with Image.open(path) as img:
            print("Image:", path)
            print("Size:", img.size)
        break
else:
    print("No supported image files found.")

Image: /content/UCSD_Anomaly_Dataset/UCSD_Anomaly_Dataset/UCSDped1/Train/Train008/145.tif
Size: (238, 158)


# Saving to Github

In [69]:
%cd "/content/drive/MyDrive/SurveillanceAnomalyDetection/repo"
!git status

/content/drive/MyDrive/SurveillanceAnomalyDetection/repo
On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .gitignore
	new file:   README.md
	new file:   notebooks/01_ucsd_dataset_setup.ipynb

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/01_ucsd_dataset_setup.ipynb

